In [1]:
import torch
import json
from model.Transformer import Transformer,encoder_stack,decoder_stack
from model.Transformer_block import encoder_block,decoder_block

with open("src_vocab.json", "r") as f:
    src_vocab = json.load(f)

with open("tgt_vocab.json", "r") as f:
    tgt_vocab = json.load(f)

embed_dim = 64

encoder_blocks= [encoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]
decoder_blocks= [decoder_block(embed_dim,1,128,eps=1e-5) for _ in range(2)]

enc_stack= encoder_stack(encoder_blocks)
dec_stack= decoder_stack(decoder_blocks)

model= Transformer(enc_stack, dec_stack, embed_dim, len(src_vocab), len(tgt_vocab))

device= torch.device("cuda" if torch.cuda.is_available() else "cpu")

model= model.to(device)

checkpoint = torch.load("checkpoint.pt", map_location=device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)



/tmp/ipykernel_68812/67041665.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("checkpoint.pt", map_location=device)


<All keys matched successfully>

In [2]:
from Data.tokenizer import Tokenizer

src_tokenizer = Tokenizer("src_vocab.json")
tgt_tokenizer = Tokenizer("tgt_vocab.json")

sentence= "hello friend how are you"

tokens= src_tokenizer.encode(sentence)
token_ids = [src_vocab["<bos>"]] + tokens + [src_vocab["<eos>"]]

src = torch.tensor(token_ids).unsqueeze(0).to(device)
tgt_in = [tgt_vocab["<bos>"]]
tgt_id= torch.tensor([tgt_in]).to(device)

EOS_id = tgt_vocab["<eos>"]
BOS_id = tgt_vocab["<bos>"]

while True:

    output= model(src,tgt_id)

    next_token_logits = output[:, -1, :]

    next_token = torch.argmax(
        next_token_logits,
        dim=-1
    )

    tgt_id = torch.cat(
        [tgt_id, next_token.unsqueeze(1)],
        dim=1
    )

    if next_token.item() == EOS_id:
        break

translated_sentence= tgt_tokenizer.decode(tgt_in)

print(translated_sentence)








<bos>
